# Frequency-Severity Model Training and Comparison - Kaggle Ready

Notebook nay duoc chuan bi de chay tren Kaggle:

- Khong phu thuoc vao package noi bo cua repo nhu `ml.scripts` hoac `app.custom_models`.
- Tu tim file CSV trong `/kaggle/input` hoac trong repo local.
- Ghi output vao `/kaggle/working/frequency_severity_notebook` tren Kaggle.
- Train va so sanh 2 model frequency, 2 model severity, sau do so sanh cac to hop frequency x severity.

Dataset can co 3 file:

- `frequency_training_dataset.csv`
- `severity_training_dataset.csv`
- `synthetic_insurance_claims.csv`

## 0. Kaggle input setup

Tren Kaggle, hay Add Data dataset chua folder `data/generated` hoac chua truc tiep 3 file CSV tren. Cell ben duoi se tu tim file theo ten.

In [ ]:
from pathlib import Path
import json
import math
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import GammaRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    import statsmodels.api as sm
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Missing statsmodels. On Kaggle, enable internet and run: !pip install statsmodels") from exc

try:
    from xgboost import XGBRegressor
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Missing xgboost. On Kaggle, enable internet and run: !pip install xgboost") from exc

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

SEED = 42
TEST_SIZE = 0.20

# Kaggle note: set MAX_ROWS=None for full data. Keep 15000 for faster notebook runs.
MAX_ROWS = 15000
CV_MAX_ROWS = 8000
CV_FOLDS = 3

IS_KAGGLE = Path("/kaggle/working").exists()
OUTPUT_DIR = Path("/kaggle/working/frequency_severity_notebook") if IS_KAGGLE else Path("ml/reports/frequency_severity_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_input_file(filename: str) -> Path:
    search_roots = []
    if IS_KAGGLE:
        search_roots.append(Path("/kaggle/input"))
    search_roots.extend([Path.cwd(), *Path.cwd().parents])

    seen = set()
    for root in search_roots:
        root = root.resolve()
        if root in seen or not root.exists():
            continue
        seen.add(root)
        direct_candidates = [
            root / filename,
            root / "data" / "generated" / filename,
        ]
        for candidate in direct_candidates:
            if candidate.exists():
                return candidate
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"Cannot find {filename}. On Kaggle, Add Data containing this CSV file."
    )

FREQUENCY_DATA = find_input_file("frequency_training_dataset.csv")
SEVERITY_DATA = find_input_file("severity_training_dataset.csv")
MASTER_DATA = find_input_file("synthetic_insurance_claims.csv")

print("Kaggle environment:", IS_KAGGLE)
print("Frequency data:", FREQUENCY_DATA)
print("Severity data:", SEVERITY_DATA)
print("Master data:", MASTER_DATA)
print("Output dir:", OUTPUT_DIR)

## 1. Feature schema, model wrappers, and metrics

In [ ]:
FREQ_TARGET = "claim_count"
FREQ_NUMERIC_FEATURES = [
    "age", "seniority_insured", "seniority_policy", "bmi", "blood_pressure",
    "prev_claim_count", "prev_claim_cost", "claim_free_years", "years_with_history",
]
FREQ_CATEGORICAL_FEATURES = [
    "type_policy", "type_policy_dg", "type_product", "reimbursement", "new_business",
    "distribution_channel", "smoker", "pre_existing_condition", "exercise_frequency",
    "occupation_risk", "prev_had_claim", "claim_free_previous_year",
]
FREQ_FEATURES = FREQ_NUMERIC_FEATURES + FREQ_CATEGORICAL_FEATURES + ["exposure_time"]

SEV_TARGET = "average_claim_severity"
SEV_NUMERIC_FEATURES = [
    "age", "seniority_insured", "bmi", "blood_pressure",
    "prev_claim_cost", "prev_claim_count", "prev_average_claim_severity",
]
SEV_CATEGORICAL_FEATURES = [
    "new_business", "type_policy", "type_policy_dg", "type_product", "reimbursement",
    "smoker", "pre_existing_condition", "exercise_frequency", "occupation_risk",
]
SEV_FEATURES = SEV_NUMERIC_FEATURES + SEV_CATEGORICAL_FEATURES

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def build_preprocessor(numeric_features, categorical_features):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ], remainder="drop")

class StatsmodelsGLMPipeline(BaseEstimator, RegressorMixin):
    def __init__(self, preprocessor, family_name="NegativeBinomial", alpha=1.0, offset_col="exposure_time"):
        self.preprocessor = preprocessor
        self.family_name = family_name
        self.alpha = alpha
        self.offset_col = offset_col
        self.results_ = None

    def fit(self, X, y):
        offset = np.log(X[self.offset_col].clip(lower=1e-6)).values if self.offset_col in X.columns else None
        X_trans = self.preprocessor.fit_transform(X)
        if hasattr(X_trans, "toarray"):
            X_trans = X_trans.toarray()
        X_trans = sm.add_constant(X_trans, has_constant="add")
        family = sm.families.NegativeBinomial(alpha=self.alpha) if self.family_name == "NegativeBinomial" else sm.families.Poisson()
        model = sm.GLM(y, X_trans, family=family, offset=offset)
        try:
            self.results_ = model.fit()
        except ValueError:
            self.results_ = model.fit(method="bfgs", maxiter=1000)
        return self

    def predict(self, X):
        offset = np.log(X[self.offset_col].clip(lower=1e-6)).values if self.offset_col in X.columns else None
        X_trans = self.preprocessor.transform(X)
        if hasattr(X_trans, "toarray"):
            X_trans = X_trans.toarray()
        X_trans = sm.add_constant(X_trans, has_constant="add")
        return self.results_.predict(X_trans, offset=offset)

class XGBoostPoissonPipeline(BaseEstimator, RegressorMixin):
    def __init__(self, preprocessor, n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, offset_col="exposure_time"):
        self.preprocessor = preprocessor
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.random_state = random_state
        self.offset_col = offset_col
        self.model_ = None

    def fit(self, X, y):
        base_margin = np.log(X[self.offset_col].clip(lower=1e-6)).values if self.offset_col in X.columns else None
        X_trans = self.preprocessor.fit_transform(X)
        if hasattr(X_trans, "toarray"):
            X_trans = X_trans.toarray()
        self.model_ = XGBRegressor(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            learning_rate=self.learning_rate,
            subsample=self.subsample,
            colsample_bytree=self.colsample_bytree,
            objective="count:poisson",
            random_state=self.random_state,
            n_jobs=-1,
        )
        self.model_.fit(X_trans, y, base_margin=base_margin)
        return self

    def predict(self, X):
        base_margin = np.log(X[self.offset_col].clip(lower=1e-6)).values if self.offset_col in X.columns else None
        X_trans = self.preprocessor.transform(X)
        if hasattr(X_trans, "toarray"):
            X_trans = X_trans.toarray()
        return self.model_.predict(X_trans, base_margin=base_margin)

class GammaGLMPipeline(BaseEstimator, RegressorMixin):
    def __init__(self, preprocessor, alpha=0.01, max_iter=1000):
        self.preprocessor = preprocessor
        self.alpha = alpha
        self.max_iter = max_iter
        self.model_ = GammaRegressor(alpha=self.alpha, max_iter=self.max_iter)

    def fit(self, X, y):
        X_trans = self.preprocessor.fit_transform(X)
        if hasattr(X_trans, "toarray"):
            X_trans = X_trans.toarray()
        self.model_.fit(X_trans, y)
        return self

    def predict(self, X):
        X_trans = self.preprocessor.transform(X)
        if hasattr(X_trans, "toarray"):
            X_trans = X_trans.toarray()
        return self.model_.predict(X_trans)

class XGBoostLognormalPipeline(BaseEstimator, RegressorMixin):
    def __init__(self, preprocessor, n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42):
        self.preprocessor = preprocessor
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.random_state = random_state
        self.model_ = None
        self.smearing_factor_ = 1.0

    def fit(self, X, y):
        log_y = np.log(pd.Series(y).clip(lower=1e-6))
        X_trans = self.preprocessor.fit_transform(X)
        if hasattr(X_trans, "toarray"):
            X_trans = X_trans.toarray()
        self.model_ = XGBRegressor(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            learning_rate=self.learning_rate,
            subsample=self.subsample,
            colsample_bytree=self.colsample_bytree,
            objective="reg:squarederror",
            random_state=self.random_state,
            n_jobs=-1,
        )
        self.model_.fit(X_trans, log_y)
        log_pred = self.model_.predict(X_trans)
        self.smearing_factor_ = float(np.mean(np.exp(log_y - log_pred)))
        return self

    def predict(self, X):
        X_trans = self.preprocessor.transform(X)
        if hasattr(X_trans, "toarray"):
            X_trans = X_trans.toarray()
        return np.exp(self.model_.predict(X_trans)) * self.smearing_factor_

In [ ]:
def normalized_gini(y_true, y_pred):
    actual = np.asarray(y_true, dtype=float)
    pred = np.asarray(y_pred, dtype=float)
    if len(actual) == 0 or np.all(actual == actual[0]):
        return 0.0

    def gini(values, scores):
        order = np.lexsort((np.arange(len(scores)), -scores))
        sorted_values = values[order]
        cumulative = np.cumsum(sorted_values)
        if cumulative[-1] == 0:
            return 0.0
        lorenz = cumulative / cumulative[-1]
        random = (np.arange(len(values)) + 1) / len(values)
        return float(np.sum(lorenz - random) / len(values))

    perfect = gini(actual, actual)
    return 0.0 if perfect == 0 else float(gini(actual, pred) / perfect)

def mean_poisson_deviance(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 1e-9)
    term = np.zeros_like(y_true)
    mask = y_true > 0
    term[mask] = y_true[mask] * np.log(y_true[mask] / y_pred[mask])
    return float(2.0 * np.mean(term - (y_true - y_pred)))

def mean_gamma_deviance(y_true, y_pred):
    y_true = np.maximum(np.asarray(y_true, dtype=float), 1e-9)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 1e-9)
    return float(np.mean(2.0 * (-np.log(y_true / y_pred) + (y_true - y_pred) / y_pred)))

def pearson_correlation(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return 0.0
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    return float(corr) if not np.isnan(corr) else 0.0

def top_10_percent_lift_frequency(y_true, y_pred, exposure):
    actual = np.asarray(y_true, dtype=float)
    pred = np.asarray(y_pred, dtype=float)
    exp = np.asarray(exposure, dtype=float)
    order = np.argsort(-(pred / np.maximum(exp, 1e-6)))
    n_top = max(1, int(len(actual) * 0.10))
    top_rate = np.sum(actual[order[:n_top]]) / max(np.sum(exp[order[:n_top]]), 1e-6)
    portfolio_rate = np.sum(actual) / max(np.sum(exp), 1e-6)
    return 0.0 if portfolio_rate == 0 else float(top_rate / portfolio_rate)

def top_10_percent_lift_cost(actual_annual_cost, predicted_annual_cost, exposure):
    actual = np.asarray(actual_annual_cost, dtype=float)
    pred = np.asarray(predicted_annual_cost, dtype=float)
    exp = np.asarray(exposure, dtype=float)
    order = np.argsort(-pred)
    n_top = max(1, int(len(actual) * 0.10))
    top_rate = np.sum(actual[order[:n_top]] * exp[order[:n_top]]) / max(np.sum(exp[order[:n_top]]), 1e-6)
    portfolio_rate = np.sum(actual * exp) / max(np.sum(exp), 1e-6)
    return 0.0 if portfolio_rate == 0 else float(top_rate / portfolio_rate)

def evaluate_frequency(y_true, y_pred, exposure):
    pred = np.maximum(np.nan_to_num(np.asarray(y_pred, dtype=float), nan=0.0, posinf=0.0, neginf=0.0), 0.0)
    return {
        "PoissonDeviance": mean_poisson_deviance(y_true, pred),
        "NormalizedGini": normalized_gini(y_true, pred),
        "Top-10% Lift": top_10_percent_lift_frequency(y_true, pred, exposure),
    }

def evaluate_severity(y_true, y_pred):
    pred = np.maximum(np.nan_to_num(np.asarray(y_pred, dtype=float), nan=0.0, posinf=0.0, neginf=0.0), 0.0)
    return {
        "MAE": float(mean_absolute_error(y_true, pred)),
        "GammaDeviance": mean_gamma_deviance(y_true, pred),
        "PearsonCorrelation": pearson_correlation(y_true, pred),
    }

## 2. Load and clean data

In [ ]:
def require_columns(df, columns, label):
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise ValueError(f"{label} is missing columns: {missing}")

def clean_frequency_frame(df):
    require_columns(df, FREQ_FEATURES + [FREQ_TARGET], "Frequency dataset")
    out = df.copy()
    for column in FREQ_NUMERIC_FEATURES + ["exposure_time", FREQ_TARGET]:
        out[column] = pd.to_numeric(out[column], errors="coerce")
    for column in FREQ_CATEGORICAL_FEATURES:
        out[column] = out[column].astype(str).str.strip()
    out = out.dropna(subset=[FREQ_TARGET, "exposure_time"])
    return out[(out[FREQ_TARGET] >= 0) & (out["exposure_time"] >= 0.1)].copy()

def clean_severity_frame(df):
    require_columns(df, SEV_FEATURES + [SEV_TARGET], "Severity dataset")
    out = df.copy()
    for column in SEV_NUMERIC_FEATURES + [SEV_TARGET]:
        out[column] = pd.to_numeric(out[column], errors="coerce")
    for column in SEV_CATEGORICAL_FEATURES:
        out[column] = out[column].astype(str).str.strip()
    out = out.dropna(subset=[SEV_TARGET])
    return out[out[SEV_TARGET] > 0].copy()

freq_df = clean_frequency_frame(pd.read_csv(FREQUENCY_DATA))
sev_df = clean_severity_frame(pd.read_csv(SEVERITY_DATA))

if MAX_ROWS is not None:
    if len(freq_df) > MAX_ROWS:
        freq_df = freq_df.sample(n=MAX_ROWS, random_state=SEED).copy()
    if len(sev_df) > MAX_ROWS:
        sev_df = sev_df.sample(n=MAX_ROWS, random_state=SEED).copy()

print(f"Frequency rows: {len(freq_df):,}")
print(f"Severity rows: {len(sev_df):,}")
display(freq_df[FREQ_FEATURES + [FREQ_TARGET]].head())
display(sev_df[SEV_FEATURES + [SEV_TARGET]].head())

## 3. Train frequency candidates

Candidates: `negative_binomial_glm` and `xgboost_poisson`. Selection metric: lowest `CV_PoissonDeviance`.

In [ ]:
def frequency_model_factory(name):
    if name == "negative_binomial_glm":
        return StatsmodelsGLMPipeline(
            build_preprocessor(FREQ_NUMERIC_FEATURES, FREQ_CATEGORICAL_FEATURES),
            family_name="NegativeBinomial",
            alpha=1.0,
        )
    if name == "xgboost_poisson":
        return XGBoostPoissonPipeline(
            build_preprocessor(FREQ_NUMERIC_FEATURES, FREQ_CATEGORICAL_FEATURES),
            random_state=SEED,
        )
    raise ValueError(name)

def cross_validate_frequency(name, df_train):
    df_cv = df_train.sample(n=min(len(df_train), CV_MAX_ROWS), random_state=SEED).copy()
    cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    dev_scores, gini_scores, lift_scores = [], [], []
    for train_idx, val_idx in cv.split(df_cv):
        fold_train = df_cv.iloc[train_idx]
        fold_val = df_cv.iloc[val_idx]
        model = frequency_model_factory(name)
        model.fit(fold_train[FREQ_FEATURES], fold_train[FREQ_TARGET])
        pred = model.predict(fold_val[FREQ_FEATURES])
        metrics = evaluate_frequency(fold_val[FREQ_TARGET], pred, fold_val["exposure_time"])
        dev_scores.append(metrics["PoissonDeviance"])
        gini_scores.append(metrics["NormalizedGini"])
        lift_scores.append(metrics["Top-10% Lift"])
    return {
        "CV_PoissonDeviance": float(np.mean(dev_scores)),
        "CV_NormalizedGini": float(np.mean(gini_scores)),
        "CV_Top-10% Lift": float(np.mean(lift_scores)),
    }

freq_train, freq_test = train_test_split(freq_df, test_size=TEST_SIZE, random_state=SEED)
frequency_candidates = {}
frequency_metrics = {}

for name in ["negative_binomial_glm", "xgboost_poisson"]:
    print(f"Training frequency candidate: {name}")
    model = frequency_model_factory(name)
    model.fit(freq_train[FREQ_FEATURES], freq_train[FREQ_TARGET])
    pred = model.predict(freq_test[FREQ_FEATURES])
    metrics = cross_validate_frequency(name, freq_train)
    metrics.update(evaluate_frequency(freq_test[FREQ_TARGET], pred, freq_test["exposure_time"]))
    frequency_candidates[name] = model
    frequency_metrics[name] = metrics

frequency_comparison = (
    pd.DataFrame(frequency_metrics).T
    .reset_index()
    .rename(columns={"index": "model"})
    .sort_values(["CV_PoissonDeviance", "PoissonDeviance"])
    .reset_index(drop=True)
)
best_frequency_model_name = frequency_comparison.iloc[0]["model"]
display(frequency_comparison)
print("Best frequency model:", best_frequency_model_name)

## 4. Train severity candidates

Candidates: `gamma_glm` and `xgboost_lognormal`. Selection metric: lowest `CV_GammaDeviance`.

In [ ]:
def severity_model_factory(name):
    if name == "gamma_glm":
        return GammaGLMPipeline(build_preprocessor(SEV_NUMERIC_FEATURES, SEV_CATEGORICAL_FEATURES))
    if name == "xgboost_lognormal":
        return XGBoostLognormalPipeline(
            build_preprocessor(SEV_NUMERIC_FEATURES, SEV_CATEGORICAL_FEATURES),
            random_state=SEED,
        )
    raise ValueError(name)

def cross_validate_severity(name, df_train):
    df_cv = df_train.sample(n=min(len(df_train), CV_MAX_ROWS), random_state=SEED).copy()
    cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    mae_scores, dev_scores, corr_scores = [], [], []
    for train_idx, val_idx in cv.split(df_cv):
        fold_train = df_cv.iloc[train_idx]
        fold_val = df_cv.iloc[val_idx]
        model = severity_model_factory(name)
        model.fit(fold_train[SEV_FEATURES], fold_train[SEV_TARGET])
        pred = model.predict(fold_val[SEV_FEATURES])
        metrics = evaluate_severity(fold_val[SEV_TARGET], pred)
        mae_scores.append(metrics["MAE"])
        dev_scores.append(metrics["GammaDeviance"])
        corr_scores.append(metrics["PearsonCorrelation"])
    return {
        "CV_MAE": float(np.mean(mae_scores)),
        "CV_GammaDeviance": float(np.mean(dev_scores)),
        "CV_PearsonCorrelation": float(np.mean(corr_scores)),
    }

sev_train, sev_test = train_test_split(sev_df, test_size=TEST_SIZE, random_state=SEED)
severity_candidates = {}
severity_metrics = {}

for name in ["gamma_glm", "xgboost_lognormal"]:
    print(f"Training severity candidate: {name}")
    model = severity_model_factory(name)
    model.fit(sev_train[SEV_FEATURES], sev_train[SEV_TARGET])
    pred = model.predict(sev_test[SEV_FEATURES])
    metrics = cross_validate_severity(name, sev_train)
    metrics.update(evaluate_severity(sev_test[SEV_TARGET], pred))
    severity_candidates[name] = model
    severity_metrics[name] = metrics

severity_comparison = (
    pd.DataFrame(severity_metrics).T
    .reset_index()
    .rename(columns={"index": "model"})
    .sort_values(["CV_GammaDeviance", "GammaDeviance"])
    .reset_index(drop=True)
)
best_severity_model_name = severity_comparison.iloc[0]["model"]
display(severity_comparison)
print("Best severity model:", best_severity_model_name)

## 5. Candidate comparison charts

In [ ]:
def barplot_metric(df, metric, title, lower_is_better=True, filename=None):
    ordered = df.sort_values(metric, ascending=lower_is_better)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.barplot(data=ordered, x="model", y=metric, ax=ax, color="#2f6f9f")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")
    for patch in ax.patches:
        value = patch.get_height()
        ax.text(patch.get_x() + patch.get_width() / 2, value, f"{value:.4f}", ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    if filename:
        fig.savefig(OUTPUT_DIR / filename, dpi=160)
    return fig

barplot_metric(frequency_comparison, "CV_PoissonDeviance", "Frequency - CV Poisson Deviance", filename="frequency_cv_poisson_deviance.png")
barplot_metric(frequency_comparison, "CV_NormalizedGini", "Frequency - CV Normalized Gini", lower_is_better=False, filename="frequency_cv_normalized_gini.png")
barplot_metric(severity_comparison, "CV_GammaDeviance", "Severity - CV Gamma Deviance", filename="severity_cv_gamma_deviance.png")
barplot_metric(severity_comparison, "CV_MAE", "Severity - CV MAE", filename="severity_cv_mae.png")
plt.show()

## 6. Combined frequency x severity evaluation

Formula:

`predicted_annual_cost = predicted_claim_count / exposure_time * predicted_severity`

In [ ]:
def combined_predictions(freq_model, sev_model, df):
    pred_claim_count = np.maximum(freq_model.predict(df[FREQ_FEATURES]), 0.0)
    pred_frequency_rate = pred_claim_count / np.maximum(df["exposure_time"].to_numpy(dtype=float), 1e-6)
    pred_severity = np.maximum(sev_model.predict(df[SEV_FEATURES]), 0.0)
    pred_annual_cost = pred_frequency_rate * pred_severity
    return pred_claim_count, pred_frequency_rate, pred_severity, pred_annual_cost

def combined_metrics(actual_annual_cost, predicted_annual_cost, exposure):
    actual = np.asarray(actual_annual_cost, dtype=float)
    pred = np.maximum(np.nan_to_num(np.asarray(predicted_annual_cost, dtype=float), nan=0.0, posinf=0.0, neginf=0.0), 0.0)
    exposure = np.asarray(exposure, dtype=float)
    actual_total = float(np.sum(actual * exposure))
    pred_total = float(np.sum(pred * exposure))
    return {
        "MAE": float(mean_absolute_error(actual, pred)),
        "NormalizedGini": float(normalized_gini(actual, pred)),
        "Top-10% Lift": top_10_percent_lift_cost(actual, pred, exposure),
        "CalibrationRatioWeighted": actual_total / pred_total if pred_total > 0 else 0.0,
        "PredMean": float(np.mean(pred)),
    }

master_df = pd.read_csv(MASTER_DATA)
require_columns(master_df, list(set(FREQ_FEATURES + SEV_FEATURES + ["annual_claim_cost", "exposure_time"])), "Master dataset")
master_df = master_df[master_df["exposure_time"] >= 0.1].copy()

if MAX_ROWS is not None and len(master_df) > MAX_ROWS:
    master_df = master_df.sample(n=MAX_ROWS, random_state=SEED).copy()

_, master_test = train_test_split(master_df, test_size=TEST_SIZE, random_state=SEED)
actual_annual_cost = master_test["annual_claim_cost"].to_numpy(dtype=float) / np.maximum(master_test["exposure_time"].to_numpy(dtype=float), 1e-6)

combined_rows = []
combined_cache = {}
for freq_name, freq_model in frequency_candidates.items():
    for sev_name, sev_model in severity_candidates.items():
        combo_name = f"{freq_name} x {sev_name}"
        pred_claim_count, pred_frequency_rate, pred_severity, pred_annual_cost = combined_predictions(freq_model, sev_model, master_test)
        pred_annual_cost = np.maximum(np.nan_to_num(pred_annual_cost, nan=0.0, posinf=0.0, neginf=0.0), 0.0)
        metrics = combined_metrics(actual_annual_cost, pred_annual_cost, master_test["exposure_time"].to_numpy(dtype=float))
        combined_rows.append({"combo": combo_name, "frequency_model": freq_name, "severity_model": sev_name, **metrics})
        combined_cache[combo_name] = {
            "pred_claim_count": pred_claim_count,
            "pred_frequency_rate": pred_frequency_rate,
            "pred_severity": pred_severity,
            "pred_annual_cost": pred_annual_cost,
        }

combined_comparison = pd.DataFrame(combined_rows).sort_values(["MAE", "CalibrationRatioWeighted"]).reset_index(drop=True)
best_combo = combined_comparison.iloc[0]["combo"]
display(combined_comparison)
print("Best combined model by MAE:", best_combo)

## 7. Actual vs predicted charts for combined model

In [ ]:
best_pred = combined_cache[best_combo]["pred_annual_cost"]
plot_df = pd.DataFrame({
    "actual_annual_cost": actual_annual_cost,
    "predicted_annual_cost": best_pred,
    "annual_claim_cost": master_test["annual_claim_cost"].to_numpy(dtype=float),
    "exposure_time": master_test["exposure_time"].to_numpy(dtype=float),
})

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    x=np.log1p(plot_df["predicted_annual_cost"]),
    y=np.log1p(plot_df["actual_annual_cost"]),
    alpha=0.30,
    s=18,
    edgecolor=None,
    ax=ax,
)
max_value = max(np.log1p(plot_df["predicted_annual_cost"]).max(), np.log1p(plot_df["actual_annual_cost"]).max())
ax.plot([0, max_value], [0, max_value], color="#dc2626", linestyle="--", linewidth=2, label="Perfect calibration")
ax.set_title(f"Actual vs Predicted Annual Cost\n{best_combo}")
ax.set_xlabel("log1p(Predicted annual claim cost)")
ax.set_ylabel("log1p(Actual annual claim cost)")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "combined_actual_vs_predicted_log1p.png", dpi=180)
plt.show()

In [ ]:
decile_df = plot_df.copy()
decile_df["decile"] = pd.qcut(decile_df["predicted_annual_cost"], 10, labels=False, duplicates="drop") + 1

decile_rows = []
for decile, group in decile_df.groupby("decile", observed=True):
    exposure_sum = np.sum(group["exposure_time"])
    decile_rows.append({
        "decile": int(decile),
        "actual": np.sum(group["annual_claim_cost"]) / exposure_sum,
        "predicted": np.sum(group["predicted_annual_cost"] * group["exposure_time"]) / exposure_sum,
    })
decile_summary = pd.DataFrame(decile_rows).sort_values("decile")

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(decile_summary))
width = 0.38
ax.bar(x - width / 2, decile_summary["actual"], width, label="Actual", color="#334155")
ax.bar(x + width / 2, decile_summary["predicted"], width, label="Predicted", color="#16a34a")
ax.set_xticks(x)
ax.set_xticklabels(decile_summary["decile"].astype(int))
ax.set_title(f"Actual vs Predicted by Predicted Risk Decile\n{best_combo}")
ax.set_xlabel("Predicted risk decile")
ax.set_ylabel("Annual claim cost per exposure")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "combined_actual_vs_predicted_deciles.png", dpi=180)
plt.show()

display(decile_summary)

## 8. Save outputs

Tren Kaggle, cac file nay nam trong `/kaggle/working/frequency_severity_notebook` va co the download tu panel Output.

In [ ]:
frequency_comparison.to_csv(OUTPUT_DIR / "frequency_candidate_comparison.csv", index=False)
severity_comparison.to_csv(OUTPUT_DIR / "severity_candidate_comparison.csv", index=False)
combined_comparison.to_csv(OUTPUT_DIR / "combined_frequency_severity_comparison.csv", index=False)

summary = {
    "best_frequency_model": best_frequency_model_name,
    "best_severity_model": best_severity_model_name,
    "best_combined_model_by_mae": best_combo,
    "max_rows": MAX_ROWS,
    "cv_max_rows": CV_MAX_ROWS,
    "test_size": TEST_SIZE,
    "seed": SEED,
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

sorted(path.name for path in OUTPUT_DIR.iterdir())